# Nanoparticle Drug-Release & Tissue-Penetration Simulator

**Development & documentation notebook (R kernel via IRkernel).**

This notebook walks through the two physics modules of the simulator, reproduces
the model-comparison and tissue-penetration figures, and demonstrates fitting the
models to experimental release data. The same functions power the Shiny app
(`app/app.R`) and the zero-install browser tool (`web/index.html`).

> Units: micrometres (µm) and hours (h) throughout.

## 0. Load the model core

All model code lives in `R/`. It uses only base R, so nothing needs installing to
run this notebook (the Shiny app additionally needs the `shiny` package).

In [ ]:
# Run from the repository root, or set the path here.
root <- '..'
for (f in c('release_models.R','tissue_diffusion.R','parameters.R','metrics.R')) {
  source(file.path(root, 'R', f))
}
p <- default_parameters()
validate_parameters(p)
str(p[c('r','h','C0','D_release','D_tissue','k_e')])

## 1. Module 1 — drug release from the nanoparticle

Six models return the cumulative fraction released, *f(t) ∈ [0, 1]*:

- **Higuchi** — `f = k_H·√t` (diffusion-controlled matrix)
- **First-order** — `f = 1 − e^(−kt)`
- **Zero-order** — `f = k·t`
- **Korsmeyer–Peppas** — `f = k·tⁿ`
- **Fickian sphere** — exact Crank series solution
- **Membrane core–shell** — reservoir device, release ∝ 1/h

In [ ]:
t <- time_grid(p)
models <- list(
  higuchi          = list(k_H = p$k_higuchi),
  first_order      = list(k = p$k_first),
  zero_order       = list(k = p$k_zero),
  korsmeyer_peppas = list(k = p$k_peppas, n = p$n_peppas),
  fickian_sphere   = list(D = p$D_release, r = p$r),
  membrane_shell   = list(D = p$D_release, r = p$r, h = p$h, K = p$K, C0 = p$C0)
)
labs <- release_model_labels()
cols <- c('#1f77b4','#d62728','#2ca02c','#9467bd','#ff7f0e','#17becf')

plot(NA, xlim=c(0,p$t_end), ylim=c(0,1), xlab='Time (h)',
     ylab='Cumulative fraction released', main='Module 1: release models')
grid()
for (i in seq_along(models)) {
  df <- release_curve(names(models)[i], t, models[[i]])
  lines(df$time, df$fraction, col=cols[i], lwd=2)
}
legend('bottomright', legend=labs[names(models)], col=cols, lwd=2, cex=0.8)

### Effect of shell thickness on the membrane-controlled model

The core–shell model is the one a formulator tunes: release rate scales as **1/h**,
so a thicker polymer coating slows release. This is the design lever the other
(empirical) models cannot express.

In [ ]:
h_values <- c(0.01, 0.02, 0.05, 0.1)
cols2 <- heat.colors(length(h_values))
plot(NA, xlim=c(0,p$t_end), ylim=c(0,1), xlab='Time (h)',
     ylab='Fraction released', main='Membrane model: thicker shell = slower release')
grid()
for (i in seq_along(h_values)) {
  f <- release_membrane_shell(t, D=p$D_release, r=p$r, h=h_values[i], K=p$K, C0=p$C0)
  lines(t, f, col=cols2[i], lwd=2.5)
}
legend('bottomright', legend=sprintf('h = %g um', h_values), col=cols2, lwd=2.5)

## 2. Module 2 — diffusion into the surrounding tissue

Fick's second law with first-order clearance, in spherical symmetry:

$$\frac{\partial C}{\partial t} = D\,\frac{1}{x^2}\frac{\partial}{\partial x}\!\left(x^2\frac{\partial C}{\partial x}\right) - k_e\,C$$

The particle-surface concentration is driven by the Module 1 release curve.

In [ ]:
# Drive the tissue source with the first-order release curve.
rel  <- release_curve('first_order', t, list(k = p$k_first))
surf <- approxfun(rel$time, p$C0 * rel$fraction, rule = 2)

sol <- solve_tissue_diffusion(
  D = p$D_tissue, r_inner = p$r, r_outer = p$r_outer,
  t_end = p$t_end, surface_conc = surf, k_e = p$k_e,
  n_x = 120, outer_bc = p$outer_bc)

snap_times <- c(2, 8, 24, 48)
cols3 <- c('#fdae61','#f46d43','#d73027','#a50026')
plot(NA, xlim=c(p$r,p$r_outer), ylim=c(0,p$C0),
     xlab='Distance from particle centre (um)', ylab='Tissue concentration',
     main='Module 2: penetration into tissue')
grid()
for (i in seq_along(snap_times)) {
  prof <- profile_at_time(sol, snap_times[i])
  lines(prof$x, prof$C, col=cols3[i], lwd=2)
}
legend('topright', legend=sprintf('t = %g h', snap_times), col=cols3, lwd=2)

cat(sprintf('Penetration depth (10%% threshold): %.2f um\n',
            penetration_depth(sol, threshold = 0.1)))

### How clearance limits penetration

Higher tissue clearance `k_e` (metabolism/perfusion) removes drug faster, so it
does not reach as far. This is a key determinant of therapeutic range.

In [ ]:
ke_values <- c(0.0, 0.1, 0.5, 1.0)
depths <- sapply(ke_values, function(ke) {
  s <- solve_tissue_diffusion(D=p$D_tissue, r_inner=p$r, r_outer=p$r_outer,
                              t_end=p$t_end, surface_conc=surf, k_e=ke, n_x=120)
  penetration_depth(s, threshold = 0.1)
})
data.frame(k_e = ke_values, penetration_depth_um = round(depths, 2))

## 3. Fit models to experimental data (with AICc model selection)

Upload your own `(time, fraction)` release measurements and let the tool pick the
best model. Here we use synthetic Korsmeyer–Peppas data with noise; the fitter
should recover it and rank it first by AICc.

In [ ]:
set.seed(42)
f_true  <- release_korsmeyer_peppas(t, k = 0.18, n = 0.55)
f_noisy <- pmin(pmax(f_true + rnorm(length(t), sd = 0.02), 0), 1)

fit <- fit_all_models(t, f_noisy)
print(fit$ranking, row.names = FALSE)
cat(sprintf('Best model by AICc: %s\n', fit$best))

best_pred <- fit$fits[[fit$best]]$predicted
plot(t, f_noisy, pch=19, col='#555555', xlab='Time (h)', ylab='Fraction released',
     main=sprintf('Best fit: %s', labs[fit$best]))
lines(t, best_pred, col='#d62728', lwd=3)
legend('bottomright', legend=c('data','best fit'), pch=c(19,NA),
       lty=c(NA,1), lwd=c(NA,3), col=c('#555555','#d62728'))

## 4. Next steps

- Turn this analysis into the interactive **Shiny app** (`app/app.R`) for the
  community, or the zero-install **browser tool** (`web/index.html`).
- Add an **ML surrogate** that maps design parameters straight to a release/
  penetration summary, skipping the PDE solve for fast design screening.

See the repository README for equations, references, and deployment notes.